In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Project : LLM Judge

In [ ]:
# Laoding datasets

train_data = pd.read_csv('/kaggle/input/llm-classification-finetuning/train.csv')
test_data = pd.read_csv('/kaggle/input/llm-classification-finetuning/test.csv')



# Todo List
1. Data exploration
2. Data analysis
3. Data cleaning
4. Data Transforming
5. Model training
6. Model testing 
7. Model evaluation

# 1. Data exploration
Here we are loading the data , taking a look on the data, identifying the categorical and numericals attributes, the explicative variables and the target variable

In [ ]:
# Train_data exploration

train_data.head()

As attended the train dataset contain the combination : model_type (a or b) , user_prompt , a_response , b_response and the result.

In [ ]:
# Selection of target variable and the explicative variables
explicative_variables = ['prompt','response_a', 'response_b']
X_train = train_data[explicative_variables]

# construction of the target variable y
conditions = [
    (train_data['winner_model_a'] == 1) ,
    (train_data['winner_model_b'] == 1) ,
    (train_data['winner_tie'] == 1) 
]
valeurs = [
    'model_a',
    'model_b',
    'egality'
]
train_data['preferences'] = np.select(conditions, valeurs, default = np.nan)

y_train = train_data['preferences']

print("The creation of the target variable succeed ")
print(y_train)

# 2. Data Analysis

Here we are just take a look on the datatypes, missing values and the significant variables for our analysis.

In [ ]:
# infos about the dataset 
train_data.info()

great any missing values 

In [ ]:
# to go on we need to know which models was questioning by the users 
models_a = train_data['model_a'].unique()
models_b = train_data['model_b'].unique()

models_a_and_b = np.concatenate([models_a , models_b])
models = pd.Series(models_a_and_b).unique()
print(models)
print('the number of used models is : ' , len(models))

64 Different models were used by the users. It is too great to be a significant attribute to help us to build our models. Instead, we will focus on user prompts and model responses to build our predictive models.

# 3. Data transforming 
Here we are dealing with a big part of our problem. We are building the necessary variable to the training of our model from the data present in the dataset.
   

In [ ]:
# dropping of the stop word in the user's prompts and in the llms responses 
# import of necessary library
import nltk
from nltk.corpus import stopwords
import pandas as pd

# nltk.download('stopwords') for the download of the stopwords library
STOP_WORDS_FR = set(stopwords.words('english'))



In [ ]:
import re
# definition of the fonction to drop stopwords in a text
def drop_stopwords(text):
    # convert the text in lower case
    text = text.lower()
    
    # drop all the number and the specifique caracters keep only the letter
    clean_text = re.sub(r'[^a-z\s]', '', text)

    # transform the text in words
    words = clean_text.split()

    # deleting the stopwords of the text
    # the stopwords are words which are't significant in the text
    empty_words = set(stopwords.words('english'))

    # filtering the significants words 
    filtered_words = [word for word in words if word not in empty_words]

    # reconstruct the text
    return " ".join(filtered_words)

# applying this method to the users prompts , the a response and the b response in the dataset
categorical_attribute = ['prompt','response_a','response_b']
for col in categorical_attribute :
    X_train[f'{col}_clean'] = X_train[col].apply(drop_stopwords)
# verification

X_train[['prompt', 'prompt_clean']].head()

It work good! Now we are entering in the stage of the vectorization of the categorical attributes by using the TD-IDF method .



# 📝 Rappel : Technique TF-IDF

Le **TF-IDF** est une technique de pondération statistique utilisée en Traitement Automatique du Langage Naturel (TALN) pour refléter l'importance d'un mot dans un document par rapport à un ensemble de documents (le corpus).

Plus la valeur TF-IDF est élevée, plus le mot est pertinent pour ce document spécifique.

---

## 1. Fréquence du Terme (Term Frequency, TF)

Le **TF** mesure la fréquence à laquelle un terme ($t$) apparaît dans un document ($d$).

$$\text{TF}(t, d) = \frac{\text{Nombre d'occurrences du terme } t \text{ dans le document } d}{\text{Nombre total de termes dans le document } d}$$

**Explication :** Si un mot apparaît souvent dans un document, il est important pour ce document. Cependant, les mots très fréquents comme "le" ou "de" auront un TF élevé mais une faible pertinence.

---

## 2. Fréquence Inverse de Document (Inverse Document Frequency, IDF)

L'**IDF** mesure l'importance générale du terme. Il pénalise les termes qui sont trop courants dans l'ensemble du corpus.

$$\text{IDF}(t, D) = \log\left(\frac{\text{Nombre total de documents } (N)}{\text{Nombre de documents contenant le terme } t}\right)$$

*Note : Souvent, on ajoute 1 au dénominateur pour éviter la division par zéro si un mot n'apparaît dans aucun document (lissage : $\log(\frac{N}{1 + \text{df}(t)})$).*

**Explication :** Si un mot apparaît dans presque tous les documents, son score IDF sera proche de zéro (car le logarithme d'une valeur proche de 1 est proche de 0). Un mot rare aura un score IDF élevé.

---

## 3. Score Final TF-IDF

Le score **TF-IDF** final est simplement le produit du TF et de l'IDF.

$$\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \text{IDF}(t, D)$$

**Interprétation :** Un mot obtient un score élevé si :

1.  Il apparaît **souvent** dans un document spécifique (TF élevé).
2.  Il apparaît **rarement** dans l'ensemble du corpus (IDF élevé).

Ceci permet de mettre en évidence les mots qui sont de bons **descripteurs** du contenu du document.

In [ ]:
from sklearn.features_extraction.text import TfidfVectorizer
import spicy.sparse # for handling of sparse matrices

# creation of the vectorizer
vectorizer = TfidfVectorizer(
    # determine the max number of words to use 
    max_feature = 5000,
    # to ignore the word that appear in less than min_df documents
    min_df = 5
)

# defining the corpus so all the cleaned messages 

cleaned_messages = ['prompt_clean', 'response_a_clean', 'response_b_clean']

# concatenate all the text documents to form the corpus 
corpus = pd.concat(X_train[col] for col in cleaned_messages)

# fit of the model
